# Where Should I Live? – Transport in 7 Schritten

**Wichtig:** Führe die Code-Zellen **von oben nach unten** mit `Shift + Enter` aus. Wenn eine Zelle einen Fehler zeigt, behebe ihn, bevor du weitermachst. Jede Zelle ruft kommentierte Funktionen in `transport_pipeline.py` auf.

Die Datei `transport_pipeline.py` enthält die ausführliche Erklärung *an den tatsächlichen Codezeilen*. Zum Öffnen einfach im Explorer links anklicken.


In [ ]:
# Diese Zelle macht den Transport-Ordner in VS Code auffindbar.
from pathlib import Path  # Hilft beim Finden von Dateien.
import sys  # Erlaubt Python, weitere Ordner zum Suchpfad hinzuzufügen.

folder = Path.cwd()  # Den aktuellen Arbeitsordner abfragen.
if not (folder / "transport").is_dir():  # Falls Jupyter bereits im Ordner transport ist.
    folder = folder.parent  # Eine Ordnerebene nach oben gehen.
sys.path.insert(0, str(folder))  # Python soll auch im GitHub-Hauptordner suchen.

from transport import transport_pipeline as tp  # Unsere vorbereiteten Funktionen laden.
print("Projekt gefunden:", folder)  # Sichtbare Bestätigung für den Start.


## Schritt 1: Winterthur → Zürich HB (Transport API)

Die Funktion `test_winterthur()` fragt bis zu vier Zugsverbindungen ab 08:00 Uhr am nächsten Dienstag ab. Sie zeigt die **Median-Fahrzeit** an (keine fix erfundene Zahl). Die API: https://transport.opendata.ch/docs.html


In [ ]:
# Teste nur die Zugverbindung Winterthur -> Zürich HB.
tp.test_winterthur()  # Anfrage an transport.opendata.ch senden und Minuten anzeigen.


## Schritt 2: Liste mit 20 Gemeinden prüfen

Die Datei `municipalities.csv` enthält 20 Orte, Bahnhöfe und Start-ZVV-Zonen. **Die letzte Spalte `route_zone_count` ist absichtlich noch leer**. Bitte mit dem offiziellen ZVV-Preisfinder die erforderlichen Zonen bis Zürich HB ermitteln, eintragen und speichern.


In [ ]:
# Die CSV-Datei mit den Gemeinden als Tabelle anschauen.
import pandas as pd  # Bibliothek für Tabellen und Datenanalyse.
places = pd.read_csv(tp.MUNICIPALITIES_FILE)  # Gemeindeliste einlesen.
display(places)  # Tabelle in Jupyter sichtbar darstellen.


## Schritt 3: ÖV-Zeiten aller 20 Orte in SQLite speichern

**Voraussetzung:** Internetzugang. Dieser Schritt macht mehrere öffentliche API-Anfragen und erstellt automatisch `transport/transport.db`. Er speichert auch die Bahnhofskoordinaten, die wir für Fahrrad und Auto brauchen.


In [ ]:
# Alle Gemeinden über die ÖV-API abfragen und in SQLite speichern.
tp.build_pt_database()  # Erstellt und füllt die SQLite-Datei transport.db.


## Schritt 4a: SBB Open Data ausprobieren

Das ist eine **Katalogsuche** nach Tarif-Datensätzen, keine automatische Ticketpreis-Abfrage. Wenn der Katalog kein passendes, frei nutzbares Preisangebot zeigt, verwenden wir die offizielle ZVV-Tabelle.


In [ ]:
# Suche nach Preisdaten im offiziellen SBB-Open-Data-Katalog.
tp.research_sbb_prices()  # Gibt Datensatznamen zur Dokumentation aus.


## Schritt 4b: Offizielle ZVV-Preise in der Datenbank speichern

Die Tabelle `zvv_pass_prices.csv` enthält Erwachsenentarife (persönlich, 2. Klasse) bis **12.12.2026**. **Zuerst `route_zone_count` manuell mit dem ZVV-Preisfinder ergänzen**, sonst bleiben die Preise pro Ort leer. Danach diese Zelle noch einmal starten.


In [ ]:
# Preise in SQLite speichern und den geprüften Streckenzonen zuordnen.
tp.update_pass_prices()  # Erstellt die ZVV-Tariftabelle und ordnet Jahrespreise zu.


## Schritt 5: Fahrrad- und Autofahrzeiten (OpenRouteService)

Registriere dich unter https://openrouteservice.org/ und erstelle einen API-Key. In der nächsten Zelle wird der Key **verdeckt** eingegeben und nicht in eine Datei geschrieben. Bitte den Key **niemals in GitHub speichern**. Dieser Schritt hat 40 mögliche Routenanfragen (20 Orte × 2 Verkehrsmittel). Bei API-Limits können einzelne Werte leer bleiben.


In [ ]:
# OpenRouteService-Key sicher eingeben (ohne ihn im Notebook anzuzeigen).
from getpass import getpass  # Versteckt die Eingabe des privaten API-Schlüssels.
ors_key = getpass("OpenRouteService API-Key: ")  # Key nur im Arbeitsspeicher halten.
tp.add_bike_car_times(ors_key)  # Fahrrad-/Autominuten und Kilometer ergänzen.


## Schritt 6: Datenbank kontrollieren

In dieser Tabelle sollten für möglichst alle Orte `pt_minutes`, `bike_minutes` und `car_minutes` stehen. Passpreise erscheinen nur, wenn die **Streckenzonen verifiziert und eingegeben** wurden.


In [ ]:
# Zeige die gespeicherten Fahrzeiten und Preise übersichtlich an.
result = tp.show_database()  # Die Datenbank in eine pandas-Tabelle laden.
display(result)  # In Jupyter sichtbar ausgeben.
print("Datensätze:", len(result))  # Prüfen, ob 20 Gemeinden gespeichert wurden.


### Fertig

Nach diesem Schritt könnt ihr `transport/transport.db`, `transport/municipalities.csv`, `transport/zvv_pass_prices.csv`, `transport/transport_pipeline.py` und dieses Notebook ins gemeinsame GitHub-Repository übernehmen. Der API-Key darf nicht im GitHub-Repository auftauchen.
